# Reciprocal Rank Fusion

You will combine ranked lists by rank based fusion without score normalization.


## RRF Core Rule

You will compute fused scores from rank positions and smoothing constant k.


In [ ]:
%pip install -q openai numpy --upgrade

In [1]:
import getpass
import os
import json
import math
import numpy as np
from collections import Counter, defaultdict
from dataclasses import dataclass, field
import re
from openai import OpenAI

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

client = OpenAI()
MODEL = "gpt-5-mini"
EMBED_MODEL = "text-embedding-3-small"

## Part 1 Basic RRF

You will implement plain RRF and inspect fused ranking output.


In [2]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[int]],
    k: int = 60
) -> list[tuple[int, float]]:
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.
    
    Args:
        ranked_lists: List of ranked lists (each list contains doc indices in rank order)
        k: Smoothing constant (default 60)
    
    Returns:
        List of (doc_idx, rrf_score) tuples sorted by score descending
    """
    rrf_scores = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            rrf_scores[doc_idx] += 1.0 / (k + rank)
    
    # Sort by RRF score descending
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Example: Fuse three ranked lists
list1 = [0, 2, 4, 1, 3]  # Rankings from BM25
list2 = [2, 0, 1, 4, 3]  # Rankings from vector search
list3 = [0, 1, 2, 3, 4]  # Rankings from another method

fused = reciprocal_rank_fusion([list1, list2, list3])

print("Individual rankings:")
print(f"  List 1 (BM25):   {list1}")
print(f"  List 2 (Vector): {list2}")
print(f"  List 3 (Other):  {list3}")
print(f"\nFused RRF ranking:")
for doc_idx, score in fused:
    print(f"  Doc {doc_idx}: RRF score = {score:.6f}")

Individual rankings:
  List 1 (BM25):   [0, 2, 4, 1, 3]
  List 2 (Vector): [2, 0, 1, 4, 3]
  List 3 (Other):  [0, 1, 2, 3, 4]

Fused RRF ranking:
  Doc 0: RRF score = 0.048916
  Doc 2: RRF score = 0.048395
  Doc 1: RRF score = 0.047627
  Doc 4: RRF score = 0.046883
  Doc 3: RRF score = 0.046394


## RRF Behavior

You will inspect rank contributions across different k values.


In [3]:
def rrf_contribution(rank: int, k: int = 60) -> float:
    """Calculate RRF contribution for a given rank."""
    return 1.0 / (k + rank)

# Show contribution decay
print("RRF contribution by rank (k=60):")
print("-" * 40)
ranks = [1, 2, 3, 5, 10, 20, 50, 100]
for rank in ranks:
    contrib = rrf_contribution(rank)
    bar = "█" * int(contrib * 1000)
    print(f"Rank {rank:3d}: {contrib:.6f} {bar}")

print("\n" + "=" * 50)
print("\nEffect of k parameter:")
print("-" * 40)

k_values = [10, 30, 60, 100]
print(f"{'Rank':<8}", end="")
for k in k_values:
    print(f"k={k:<6}", end="")
print()

for rank in [1, 5, 10, 50]:
    print(f"{rank:<8}", end="")
    for k in k_values:
        print(f"{rrf_contribution(rank, k):.4f}  ", end="")
    print()

RRF contribution by rank (k=60):
----------------------------------------
Rank   1: 0.016393 ████████████████
Rank   2: 0.016129 ████████████████
Rank   3: 0.015873 ███████████████
Rank   5: 0.015385 ███████████████
Rank  10: 0.014286 ██████████████
Rank  20: 0.012500 ████████████
Rank  50: 0.009091 █████████
Rank 100: 0.006250 ██████


Effect of k parameter:
----------------------------------------
Rank    k=10    k=30    k=60    k=100   
1       0.0909  0.0323  0.0164  0.0099  
5       0.0667  0.0286  0.0154  0.0095  
10      0.0500  0.0250  0.0143  0.0091  
50      0.0167  0.0125  0.0091  0.0067  


## Part 2 Weighted RRF

You will add list specific weighting.


In [4]:
def weighted_rrf(
    ranked_lists: list[list[int]],
    weights: list[float] = None,
    k: int = 60
) -> list[tuple[int, float]]:
    """
    RRF with optional weights for each ranked list.
    
    Example use case:
    - Original query results: weight = 2.0
    - Expanded query results: weight = 1.0
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    
    rrf_scores = defaultdict(float)
    
    for weight, ranked_list in zip(weights, ranked_lists):
        for rank, doc_idx in enumerate(ranked_list, start=1):
            rrf_scores[doc_idx] += weight / (k + rank)
    
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Compare weighted vs unweighted
original_query_results = [0, 2, 4]
expanded_query_results = [3, 2, 1]

print("Unweighted RRF:")
unweighted = reciprocal_rank_fusion([original_query_results, expanded_query_results])
for doc, score in unweighted[:5]:
    print(f"  Doc {doc}: {score:.6f}")

print("\nWeighted RRF (original query 2x):")
weighted = weighted_rrf([original_query_results, expanded_query_results], weights=[2.0, 1.0])
for doc, score in weighted[:5]:
    print(f"  Doc {doc}: {score:.6f}")

Unweighted RRF:
  Doc 2: 0.032258
  Doc 0: 0.016393
  Doc 3: 0.016393
  Doc 4: 0.015873
  Doc 1: 0.015873

Weighted RRF (original query 2x):
  Doc 2: 0.048387
  Doc 0: 0.032787
  Doc 4: 0.031746
  Doc 3: 0.016393
  Doc 1: 0.015873


## Part 3 Top Rank Bonus

You will preserve high confidence top hits with bonus logic.


In [5]:
def rrf_with_top_rank_bonus(
    ranked_lists: list[list[int]],
    k: int = 60,
    bonus_rank_1: float = 0.05,
    bonus_rank_2_3: float = 0.02
) -> list[tuple[int, float]]:
    """
    RRF with bonus for documents ranking in top positions.
    
    This preserves high-confidence matches that might otherwise
    be diluted by expanded queries.
    """
    rrf_scores = defaultdict(float)
    bonuses = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            # Standard RRF contribution
            rrf_scores[doc_idx] += 1.0 / (k + rank)
            
            # Track top-rank bonuses (keep highest)
            if rank == 1:
                bonuses[doc_idx] = max(bonuses[doc_idx], bonus_rank_1)
            elif rank in [2, 3]:
                bonuses[doc_idx] = max(bonuses[doc_idx], bonus_rank_2_3)
    
    # Apply bonuses
    final_scores = {}
    for doc_idx, rrf_score in rrf_scores.items():
        final_scores[doc_idx] = rrf_score + bonuses.get(doc_idx, 0)
    
    sorted_results = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results

# Example showing bonus effect
lists = [
    [5, 2, 3, 1, 4],  # Doc 5 is #1 here
    [1, 3, 5, 2, 4],  # Doc 1 is #1 here
    [2, 1, 3, 4, 5],  # Doc 2 is #1 here
]

print("Without top-rank bonus:")
standard = reciprocal_rank_fusion(lists)
for doc, score in standard[:5]:
    print(f"  Doc {doc}: {score:.6f}")

print("\nWith top-rank bonus (+0.05 for #1, +0.02 for #2-3):")
with_bonus = rrf_with_top_rank_bonus(lists)
for doc, score in with_bonus[:5]:
    print(f"  Doc {doc}: {score:.6f}")

Without top-rank bonus:
  Doc 2: 0.048147
  Doc 1: 0.048147
  Doc 3: 0.047875
  Doc 5: 0.047651
  Doc 4: 0.046394

With top-rank bonus (+0.05 for #1, +0.02 for #2-3):
  Doc 2: 0.098147
  Doc 1: 0.098147
  Doc 5: 0.097651
  Doc 3: 0.067875
  Doc 4: 0.046394


## Part 4 LLM Reranking

You will rerank fused candidates with model based relevance scores.


In [6]:
def rerank_with_llm(
    query: str,
    documents: list[str],
    doc_indices: list[int]
) -> list[tuple[int, float]]:
    """
    Rerank documents using LLM relevance scoring.
    
    Returns list of (doc_idx, relevance_score) tuples.
    Score is 0.0-1.0 based on LLM assessment.
    """
    results = []
    
    for doc_idx in doc_indices:
        doc_text = documents[doc_idx][:500]  # Truncate for efficiency
        
        prompt = f"""Rate the relevance of this document to the query.

Query: {query}
Document: {doc_text}

Is this document relevant to the query? Answer with just "yes" or "no", then a confidence score from 0-10.
Format: yes/no, score
Example: yes, 8"""
        
        response = client.responses.create(
            model=MODEL,
            input=prompt
        )
        
        # Parse response
        text = response.output_text.lower().strip()
        try:
            parts = text.split(",")
            is_relevant = "yes" in parts[0]
            score = int(parts[1].strip()) / 10.0
            
            # If not relevant, reduce score
            if not is_relevant:
                score *= 0.3
                
            results.append((doc_idx, score))
        except:
            # Fallback if parsing fails
            results.append((doc_idx, 0.5))
    
    return results

# Test reranking
test_docs = [
    "Python is a programming language known for readability.",
    "Machine learning uses algorithms to find patterns in data.",
    "The Python snake is found in tropical regions.",
    "Deep learning neural networks require large datasets.",
    "JavaScript is used for web development.",
]

query = "Python programming tutorials"
candidates = [0, 1, 2, 3, 4]  # All docs as candidates

print(f"Query: '{query}'")
print("\nLLM Reranking scores:")
reranked = rerank_with_llm(query, test_docs, candidates)
reranked.sort(key=lambda x: x[1], reverse=True)

for doc_idx, score in reranked:
    print(f"  [{score:.2f}] Doc {doc_idx}: {test_docs[doc_idx][:50]}...")

Query: 'Python programming tutorials'

LLM Reranking scores:
  [0.30] Doc 1: Machine learning uses algorithms to find patterns ...
  [0.30] Doc 2: The Python snake is found in tropical regions....
  [0.30] Doc 3: Deep learning neural networks require large datase...
  [0.30] Doc 4: JavaScript is used for web development....
  [0.24] Doc 0: Python is a programming language known for readabi...


## Part 5 Position Aware Blending

You will blend fusion and reranker scores by rank zones.


In [7]:
def position_aware_blend(
    rrf_results: list[tuple[int, float]],
    rerank_results: list[tuple[int, float]]
) -> list[tuple[int, float, dict]]:
    """
    Blend RRF and reranker scores with position-aware weighting.
    
    Returns:
        List of (doc_idx, final_score, breakdown) tuples
    """
    # Create lookup dictionaries
    rrf_by_doc = {doc: (rank, score) for rank, (doc, score) in enumerate(rrf_results, 1)}
    rerank_by_doc = {doc: score for doc, score in rerank_results}
    
    # Normalize RRF scores to 0-1
    max_rrf = max(score for _, score in rrf_results) if rrf_results else 1
    
    results = []
    for doc_idx, rrf_score in rrf_results:
        rrf_rank = rrf_by_doc[doc_idx][0]
        rrf_normalized = rrf_score / max_rrf
        rerank_score = rerank_by_doc.get(doc_idx, 0.5)
        
        # Position-aware blending weights
        if rrf_rank <= 3:
            rrf_weight = 0.75
            blend_type = "top"
        elif rrf_rank <= 10:
            rrf_weight = 0.60
            blend_type = "mid"
        else:
            rrf_weight = 0.40
            blend_type = "low"
        
        rerank_weight = 1 - rrf_weight
        
        # Compute blended score
        final_score = rrf_weight * rrf_normalized + rerank_weight * rerank_score
        
        results.append((doc_idx, final_score, {
            "rrf_rank": rrf_rank,
            "rrf_score": rrf_normalized,
            "rerank_score": rerank_score,
            "blend_type": blend_type,
            "rrf_weight": rrf_weight
        }))
    
    # Sort by final score
    results.sort(key=lambda x: x[1], reverse=True)
    return results

## Part 6 Complete Pipeline

You will run expansion, retrieval, fusion, reranking, and blending together.


In [8]:
@dataclass
class AdvancedRetriever:
    """
    Complete retrieval pipeline with:
    1. Query expansion
    2. Parallel BM25 + Vector retrieval
    3. RRF fusion with top-rank bonus
    4. LLM reranking
    5. Position-aware blending
    """
    k: int = 60
    top_k_candidates: int = 30
    
    def __post_init__(self):
        self.documents = []
        self.bm25_index = None
        self.embeddings = []
    
    def index(self, documents: list[str]):
        """Index documents for retrieval."""
        self.documents = documents
        
        # BM25 indexing (simplified - reuse our implementation)
        self.bm25_index = self._build_bm25_index(documents)
        
        # Vector indexing
        print("Computing embeddings...")
        response = client.embeddings.create(
            model=EMBED_MODEL,
            input=documents
        )
        self.embeddings = [item.embedding for item in response.data]
        print(f"Indexed {len(documents)} documents")
    
    def _build_bm25_index(self, documents):
        """Build BM25 index."""
        # Tokenize
        def tokenize(text):
            return [t.lower() for t in re.findall(r'\b\w+\b', text) if len(t) > 1]
        
        doc_freqs = [Counter(tokenize(doc)) for doc in documents]
        doc_lengths = [sum(freq.values()) for freq in doc_freqs]
        avgdl = sum(doc_lengths) / len(doc_lengths)
        
        # IDF
        doc_containing = Counter()
        for freq in doc_freqs:
            for term in freq:
                doc_containing[term] += 1
        
        n = len(documents)
        idf = {term: math.log((n - df + 0.5) / (df + 0.5) + 1) 
               for term, df in doc_containing.items()}
        
        return {"doc_freqs": doc_freqs, "doc_lengths": doc_lengths, 
                "avgdl": avgdl, "idf": idf, "tokenize": tokenize}
    
    def _bm25_search(self, query: str, top_k: int = 100) -> list[int]:
        """BM25 search returning ranked doc indices."""
        idx = self.bm25_index
        query_terms = idx["tokenize"](query)
        k1, b = 1.5, 0.75
        
        scores = []
        for i, doc_freq in enumerate(idx["doc_freqs"]):
            score = 0
            for term in query_terms:
                if term in idx["idf"]:
                    tf = doc_freq.get(term, 0)
                    idf = idx["idf"][term]
                    dl = idx["doc_lengths"][i]
                    score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / idx["avgdl"]))
            scores.append((i, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [i for i, s in scores[:top_k] if s > 0]
    
    def _vector_search(self, query: str, top_k: int = 100) -> list[int]:
        """Vector search returning ranked doc indices."""
        response = client.embeddings.create(model=EMBED_MODEL, input=query)
        query_emb = np.array(response.data[0].embedding)
        
        scores = []
        for i, doc_emb in enumerate(self.embeddings):
            doc_emb = np.array(doc_emb)
            sim = np.dot(query_emb, doc_emb) / (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb))
            scores.append((i, sim))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [i for i, _ in scores[:top_k]]
    
    def _expand_query(self, query: str) -> list[str]:
        """Generate query expansions."""
        prompt = f"""Generate 2 alternative search queries for: {query}
Return only a JSON array of strings."""
        
        response = client.responses.create(model=MODEL, input=prompt)
        try:
            return json.loads(response.output_text)
        except:
            return []
    
    def search(self, query: str, top_k: int = 10, verbose: bool = False) -> list[dict]:
        """Full retrieval pipeline."""
        
        # Step 1: Query expansion
        queries = [query] + self._expand_query(query)
        if verbose:
            print(f"Queries: {queries}")
        
        # Step 2: Parallel retrieval
        ranked_lists = []
        weights = []
        
        for i, q in enumerate(queries):
            # BM25
            bm25_results = self._bm25_search(q)
            ranked_lists.append(bm25_results)
            weights.append(2.0 if i == 0 else 1.0)  # Original query gets 2x weight
            
            # Vector
            vector_results = self._vector_search(q)
            ranked_lists.append(vector_results)
            weights.append(2.0 if i == 0 else 1.0)
        
        # Step 3: RRF fusion with top-rank bonus
        rrf_scores = defaultdict(float)
        bonuses = defaultdict(float)
        
        for weight, ranked_list in zip(weights, ranked_lists):
            for rank, doc_idx in enumerate(ranked_list, start=1):
                rrf_scores[doc_idx] += weight / (self.k + rank)
                if rank == 1:
                    bonuses[doc_idx] = max(bonuses[doc_idx], 0.05)
                elif rank in [2, 3]:
                    bonuses[doc_idx] = max(bonuses[doc_idx], 0.02)
        
        # Apply bonuses and sort
        rrf_results = [(doc, score + bonuses.get(doc, 0)) 
                       for doc, score in rrf_scores.items()]
        rrf_results.sort(key=lambda x: x[1], reverse=True)
        
        # Step 4: Take top candidates for reranking
        candidates = [doc for doc, _ in rrf_results[:self.top_k_candidates]]
        
        if verbose:
            print(f"RRF top candidates: {candidates[:10]}")
        
        # Step 5: LLM reranking
        rerank_results = rerank_with_llm(query, self.documents, candidates)
        
        # Step 6: Position-aware blending
        blended = position_aware_blend(rrf_results[:self.top_k_candidates], rerank_results)
        
        # Format results
        final_results = []
        for doc_idx, score, breakdown in blended[:top_k]:
            final_results.append({
                "doc_idx": doc_idx,
                "score": score,
                "text": self.documents[doc_idx],
                "breakdown": breakdown
            })
        
        return final_results

In [9]:
# Test the complete pipeline
corpus = [
    "Python is a high-level programming language emphasizing code readability and simplicity.",
    "Machine learning enables computers to learn from data without explicit programming.",
    "Deep learning uses neural networks with many layers to model complex patterns.",
    "Natural language processing helps computers understand and generate human language.",
    "SQL is a domain-specific language for managing relational databases.",
    "NoSQL databases provide flexible schemas for unstructured data storage.",
    "Docker containers package applications with dependencies for deployment.",
    "Kubernetes orchestrates containerized applications across clusters.",
    "Git enables version control and collaboration on source code.",
    "Agile methodology focuses on iterative development and customer feedback.",
    "REST APIs use HTTP methods for communication between systems.",
    "GraphQL provides a query language for APIs with strong typing.",
]

retriever = AdvancedRetriever()
retriever.index(corpus)

Computing embeddings...
Indexed 12 documents


In [10]:
# Run a search
query = "how to store and query data"
results = retriever.search(query, top_k=5, verbose=True)

print(f"\nResults for: '{query}'")
print("=" * 70)

for i, result in enumerate(results, 1):
    b = result["breakdown"]
    print(f"\n{i}. [Score: {result['score']:.3f}] (RRF rank: {b['rrf_rank']}, blend: {b['blend_type']})")
    print(f"   RRF: {b['rrf_score']:.3f} × {b['rrf_weight']:.0%} + Rerank: {b['rerank_score']:.3f} × {1-b['rrf_weight']:.0%}")
    print(f"   {result['text'][:80]}...")

Queries: ['how to store and query data', 'best practices for storing and querying data in databases', 'compare data storage and query options: SQL vs NoSQL vs search engines']
RRF top candidates: [5, 1, 11, 3, 9, 8, 4, 0, 2, 10]

Results for: 'how to store and query data'

1. [Score: 0.925] (RRF rank: 1, blend: top)
   RRF: 1.000 × 75% + Rerank: 0.700 × 25%
   NoSQL databases provide flexible schemas for unstructured data storage....

2. [Score: 0.803] (RRF rank: 2, blend: top)
   RRF: 0.980 × 75% + Rerank: 0.270 × 25%
   Machine learning enables computers to learn from data without explicit programmi...

3. [Score: 0.789] (RRF rank: 7, blend: mid)
   RRF: 0.648 × 60% + Rerank: 1.000 × 40%
   SQL is a domain-specific language for managing relational databases....

4. [Score: 0.758] (RRF rank: 3, blend: top)
   RRF: 0.810 × 75% + Rerank: 0.600 × 25%
   GraphQL provides a query language for APIs with strong typing....

5. [Score: 0.519] (RRF rank: 4, blend: mid)
   RRF: 0.664 × 60% + Rer

## Exercises With Starter and Solution

### Exercise 1

You will make k adaptive to retrieval context size.


In [11]:
# Starter code for Exercise 1
def adaptive_k(num_lists: int, avg_list_length: int) -> int:
    # TODO map list count and list length to stable k
    pass


In [12]:
# Solution for Exercise 1
def adaptive_k(num_lists: int, avg_list_length: int) -> int:
    base = 20
    list_component = 4 * num_lists
    length_component = int(0.35 * avg_list_length)
    k = base + list_component + length_component
    return max(20, min(120, k))

for n_lists, avg_len in [(2, 20), (6, 60), (10, 120)]:
    print(n_lists, avg_len, '->', adaptive_k(n_lists, avg_len))


2 20 -> 35
6 60 -> 65
10 120 -> 102


### Exercise 2

You will blend reranker scores with confidence weighting.


In [13]:
# Starter code for Exercise 2
def confidence_weighted_blend(
    rrf_results: list[tuple[int, float]],
    rerank_results: list[tuple[int, float, float]]
) -> list[tuple[int, float]]:
    # TODO use confidence to control reranker influence
    pass


In [14]:
# Solution for Exercise 2
def confidence_weighted_blend(
    rrf_results: list[tuple[int, float]],
    rerank_results: list[tuple[int, float, float]]
) -> list[tuple[int, float]]:
    if not rrf_results:
        return []

    max_rrf = max(score for _, score in rrf_results) or 1.0
    rerank_by_doc = {doc: (score, confidence) for doc, score, confidence in rerank_results}

    blended = []
    for doc, rrf_score in rrf_results:
        rrf_norm = rrf_score / max_rrf
        rerank_score, confidence = rerank_by_doc.get(doc, (0.5, 0.5))

        rerank_weight = 0.2 + 0.6 * float(confidence)
        final_score = (1 - rerank_weight) * rrf_norm + rerank_weight * rerank_score
        blended.append((doc, final_score))

    blended.sort(key=lambda x: x[1], reverse=True)
    return blended

sample_rrf = [(0, 0.09), (1, 0.08), (2, 0.07)]
sample_rerank = [(0, 0.6, 0.3), (1, 0.9, 0.8), (2, 0.7, 0.5)]
print(confidence_weighted_blend(sample_rrf, sample_rerank))


[(1, 0.8964444444444445), (0, 0.848), (2, 0.7388888888888889)]


### Exercise 3

You will implement two stage cascaded reranking.


In [15]:
# Starter code for Exercise 3
def cascaded_rerank(
    query: str,
    documents: list[str],
    candidates: list[int],
    stage1_keep: int = 10
) -> list[tuple[int, float]]:
    # TODO add fast stage and precise stage
    pass


In [16]:
# Solution for Exercise 3
def cascaded_rerank(
    query: str,
    documents: list[str],
    candidates: list[int],
    stage1_keep: int = 10
) -> list[tuple[int, float]]:
    query_terms = set(re.findall(r'\b\w+\b', query.lower()))

    # Stage 1 fast lexical filter
    stage1 = []
    for idx in candidates:
        doc_terms = set(re.findall(r'\b\w+\b', documents[idx].lower()))
        overlap = len(query_terms & doc_terms)
        stage1.append((idx, overlap))

    stage1.sort(key=lambda x: x[1], reverse=True)
    shortlist = [idx for idx, _ in stage1[:stage1_keep]]

    # Stage 2 model reranking
    reranked = rerank_with_llm(query, documents, shortlist)
    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked

demo_docs = [
    'Python tutorials for beginners',
    'Snake habitat in tropical forests',
    'Advanced Python type hints',
    'Web backend with FastAPI',
]
print(cascaded_rerank('Python backend tutorials', demo_docs, [0, 1, 2, 3], stage1_keep=3))


[(3, 1.0), (0, 0.7), (2, 0.21)]


## What You Learned

You implemented production style fusion and reranking controls.
